# Text-to-Text (TTT) — LLM benchmark on Apple Silicon

Runs every prompt of a prompt set through a model with **MLX** (`mlx-lm`) in **float16** and records, per prompt:
input / output tokens, output words, first token, time to first token, prompt processing rate, inference rate and
inference time, plus system metrics (CPU / GPU / NPU utilisation, power, temperature) sampled by `macmon`.

Each model is converted once from its original Hugging Face checkpoint into a local float16 MLX copy at
`models/<model>-fp16/`, then loaded from there.

**Outputs**
- `outputs/ttt/<family>/<size>/<params>/<prompt set>/`: one response file per prompt and `<model>_simple_metrics.json`
- `metrics_analysis/ttt/<family>/<size>/<params>/<prompt set>/`: `Model_Level_Metrics/<model>_prompt_metrics.csv`
  and `System_Level_Metrics/<prompt id>_system_metrics.csv`

## 1. Settings and model registry

In [ ]:
import gc
import json
import os
import sys
import tempfile
import time
from pathlib import Path

import mlx.core as mx
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'prompts').is_dir())   # project root
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

# family     size      params  model                         original checkpoint (Hugging Face, ungated)  params (B)
REGISTRY = [
    ('Phi-3',   'SMALL',  '3.8B', 'Phi-3-mini-128k-instruct',   'microsoft/Phi-3-mini-128k-instruct',      3.82),
    ('Phi-3',   'MEDIUM', '7B',   'Phi-3-small-128k-instruct',  'microsoft/Phi-3-small-128k-instruct',     7.39),
    ('Phi-3',   'LARGE',  '14B',  'Phi-3-medium-128k-instruct', 'microsoft/Phi-3-medium-128k-instruct',   13.96),
    ('Llama-3', 'SMALL',  '1B',   'Llama-3.2-1B-Instruct',      'unsloth/Llama-3.2-1B-Instruct',           1.24),
    ('Llama-3', 'SMALL',  '3B',   'Llama-3.2-3B-Instruct',      'unsloth/Llama-3.2-3B-Instruct',           3.21),
    ('Llama-3', 'MEDIUM', '8B',   'Llama-3.1-8B-Instruct',      'unsloth/Llama-3.1-8B-Instruct',           8.03),
    ('Llama-3', 'LARGE',  '70B',  'Llama-3.3-70B-Instruct',     'unsloth/Llama-3.3-70B-Instruct',         70.55),
    ('Falcon3', 'SMALL',  '1B',   'Falcon3-1B-Instruct',        'tiiuae/Falcon3-1B-Instruct',              1.67),
    ('Falcon3', 'SMALL',  '3B',   'Falcon3-3B-Instruct',        'tiiuae/Falcon3-3B-Instruct',              3.23),
    ('Falcon3', 'MEDIUM', '7B',   'Falcon3-7B-Instruct',        'tiiuae/Falcon3-7B-Instruct',              7.46),
    ('Falcon3', 'LARGE',  '10B',  'Falcon3-10B-Instruct',       'tiiuae/Falcon3-10B-Instruct',            10.31),
    ('Qwen2.5', 'SMALL',  '0.5B', 'Qwen2.5-0.5B-Instruct',      'Qwen/Qwen2.5-0.5B-Instruct',              0.49),
    ('Qwen2.5', 'SMALL',  '1.5B', 'Qwen2.5-1.5B-Instruct',      'Qwen/Qwen2.5-1.5B-Instruct',              1.54),
    ('Qwen2.5', 'SMALL',  '3B',   'Qwen2.5-3B-Instruct',        'Qwen/Qwen2.5-3B-Instruct',                3.09),
    ('Qwen2.5', 'MEDIUM', '7B',   'Qwen2.5-7B-Instruct',        'Qwen/Qwen2.5-7B-Instruct',                7.62),
    ('Qwen2.5', 'MEDIUM', '14B',  'Qwen2.5-14B-Instruct',       'Qwen/Qwen2.5-14B-Instruct',              14.77),
    ('Qwen2.5', 'LARGE',  '32B',  'Qwen2.5-32B-Instruct',       'Qwen/Qwen2.5-32B-Instruct',              32.76),
    ('Qwen2.5', 'LARGE',  '72B',  'Qwen2.5-72B-Instruct',       'Qwen/Qwen2.5-72B-Instruct',              72.71),
    ('Llama-2', 'SMALL',  '7B',   'Llama-2-7b-hf',              'NousResearch/Llama-2-7b-hf',              6.74),   # base (not chat) models
    ('Llama-2', 'MEDIUM', '13B',  'Llama-2-13b-hf',             'NousResearch/Llama-2-13b-hf',            13.02),
    ('Llama-2', 'LARGE',  '70B',  'Llama-2-70b-hf',             'NousResearch/Llama-2-70b-hf',            68.98),
]
MODELS = pd.DataFrame(REGISTRY, columns=['family', 'size', 'params', 'model', 'source', 'params_b']).set_index('model')
MODELS['path'] = 'models/' + MODELS.index + '-fp16'           # local float16 MLX copy of each model
MODELS['fp16_gb'] = (MODELS.params_b * 2).round(1)             # 2 bytes per parameter
GPU_BUDGET_GB = mx.device_info()['max_recommended_working_set_size'] / 1e9
MODELS['fits'] = MODELS.fp16_gb < 0.9 * GPU_BUDGET_GB          # leave room for the KV cache and macOS

# Pick the model here, or from the terminal with TTT_FAMILY / TTT_SIZE / TTT_PARAMS / TTT_PROMPTS / TTT_COMPARE
FAMILY = os.environ.get('TTT_FAMILY', 'Llama-3')   # Phi-3 | Llama-3 | Falcon3 | Qwen2.5 | Llama-2
SIZE = os.environ.get('TTT_SIZE', 'MEDIUM')         # SMALL | MEDIUM | LARGE
PARAMS = os.environ.get('TTT_PARAMS') or MODELS.query('family == @FAMILY and size == @SIZE').params.iloc[0]
MODEL = MODELS.query('family == @FAMILY and size == @SIZE and params == @PARAMS').index[0]
PROMPT_SET = os.environ.get('TTT_PROMPTS', 'ttt_prompts_500w')   # prompts/tti_tts_ttt_prompts/<set>.csv (id, prompt)
MAX_NEW_TOKENS = min(int(os.environ.get('TTT_MAX_TOKENS', 1024)), 1024)   # hard cap of 1,024 output tokens
COMPARE_FAMILY = os.environ.get('TTT_COMPARE', '1') == '1'       # section 5: also run the family's other models

print(f'Selected: {FAMILY} / {SIZE} / {PARAMS} -> {MODEL} -> {MODELS.path[MODEL]}')
print(f'GPU memory budget: {GPU_BUDGET_GB:.1f} GB')
MODELS[MODELS.family == FAMILY][['size', 'params', 'source', 'path', 'fp16_gb', 'fits']]

## 2. Load a model

The first time a model is used, its original checkpoint is downloaded and converted to a float16 MLX model in
`models/<model>-fp16/` (one-time step); after that it loads from disk. A model whose float16 weights don't fit in
this Mac's GPU memory stops with a clear error.

In [ ]:
from huggingface_hub import snapshot_download
from mlx_lm import convert, load, stream_generate
from mlx_lm.sample_utils import make_sampler

TRUST_REMOTE_CODE = {'Phi-3-small-128k-instruct'}   # ships its own (tiktoken) tokenizer code
STOP_TOKENS = {'Phi-3-mini-128k-instruct': '<|end|>', 'Phi-3-medium-128k-instruct': '<|end|>'}   # end of a Phi-3 turn

def build_fp16(name):
    # Convert the original checkpoint to a float16 MLX model at MODELS.path (first use only)
    m = MODELS.loc[name]
    print(f'Converting {m.source} -> {m.path} (float16), one-time step...')
    src = Path(snapshot_download(m.source, token=False,
                                 allow_patterns=['*.json', '*.safetensors', '*.py', '*.model', '*.tiktoken', '*.txt']))
    with tempfile.TemporaryDirectory() as tmp:
        cfg = json.loads((src / 'config.json').read_text())
        if cfg['model_type'] == 'phi3' and cfg.get('rope_scaling', {}).get('type') == 'su':
            # Phi-3-medium's config uses the legacy rope type 'su', which transformers 5 rejects:
            # convert from a linked copy (no weights copied) with the rope settings in the current format
            for f in src.iterdir():
                (Path(tmp) / f.name).symlink_to(f.resolve())
            cfg['rope_scaling'].update(type='longrope', original_max_position_embeddings=cfg['original_max_position_embeddings'])
            (Path(tmp) / 'config.json').unlink()
            (Path(tmp) / 'config.json').write_text(json.dumps(cfg, indent=2))
            src = Path(tmp)
        convert(hf_path=str(src), mlx_path=m.path, dtype='float16', trust_remote_code=name in TRUST_REMOTE_CODE)

def load_model(name):
    m = MODELS.loc[name]
    if not m.fits:
        raise MemoryError(f"{name} needs {m.fp16_gb} GB in float16, more than this Mac's GPU can use ({GPU_BUDGET_GB:.1f} GB)")
    if not Path(m.path).exists():
        build_fp16(name)
    t0 = time.perf_counter()
    model, tok = load(m.path, tokenizer_config={'trust_remote_code': True} if name in TRUST_REMOTE_CODE else None)
    template = Path(m.path) / 'chat_template.jinja'
    if tok.chat_template is None and template.exists():   # Phi-3-small's custom tokenizer doesn't read this file
        tok._tokenizer.chat_template = template.read_text()
    if name in STOP_TOKENS:
        tok.add_eos_token(STOP_TOKENS[name])
    return model, tok, time.perf_counter() - t0

## 3. Prompts

Each prompt is sent as its own conversation in the model's chat template. Base models (Llama-2) have no chat template,
so the request is written as an instruction / response document that the model continues.

In [ ]:
PROMPTS = pd.read_csv(Path('prompts/tti_tts_ttt_prompts') / f'{PROMPT_SET}.csv', dtype={'id': str})
PROMPTS['id'] = PROMPTS.id.str.zfill(2)
BASE_STOP = '\n### '   # a base model starting a new section has finished its answer

def encode(tok, text):
    if tok.chat_template is None:
        return tok.encode(f'Below is an instruction, followed by a response that completes it.\n\n'
                          f'### Instruction:\n{text}\n\n### Response:\n')
    prompt = tok.apply_chat_template([{'role': 'user', 'content': text}], add_generation_prompt=True, tokenize=False)
    return tok.encode(prompt, add_special_tokens=False)

print(f'{len(PROMPTS)} prompts from {PROMPT_SET}.csv')
PROMPTS

## 4. Run the selected model

Greedy decoding (temperature 0), so results are reproducible. Each prompt is wrapped in `metrics_monitor_start()` /
`metrics_monitor_stop()` (from `macmon_metrics/`), which saves that prompt's system metrics every 100 ms.
The summary JSON averages the per-prompt metrics and the system metrics over the whole run.

In [ ]:
import macmon_metrics.metrics_monitor as mm
from macmon_metrics.metrics_monitor import metrics_monitor_start, metrics_monitor_stop, setup_output_directory

if mm._tracker is None:   # start macmon now, so every prompt's capture has samples from its first moment
    mm._tracker = mm.MacMonTracker(binary_path=mm._resolve_macmon_binary())
    mm._tracker.start()
    time.sleep(1.5)

def run_model(name):
    # Run every prompt through one model; save responses, per-prompt metrics and a summary
    m = MODELS.loc[name]
    sub = Path(m.family) / m['size'] / m.params / PROMPT_SET
    out_dir = Path('outputs/ttt') / sub
    out_dir.mkdir(parents=True, exist_ok=True)
    sys_dir, model_dir = setup_output_directory(str(Path('metrics_analysis/ttt') / sub))

    model, tok, load_s = load_model(name)
    for _ in stream_generate(model, tok, tok.encode('Hello'), max_tokens=8):   # warm-up: compile GPU kernels
        pass
    rows, hit_limit, peak_gb = [], 0, 0.0
    for p in PROMPTS.itertuples():
        ids = encode(tok, p.prompt)
        metrics_monitor_start(idx=p.id, metrics_folder=sys_dir)
        text, first, t0 = '', None, time.perf_counter()
        for r in stream_generate(model, tok, ids, max_tokens=MAX_NEW_TOKENS, sampler=make_sampler(temp=0.0)):
            if first is None:
                first, ttft = r.text, time.perf_counter() - t0
            text += r.text
            if tok.chat_template is None and BASE_STOP in text.lstrip():
                text = text.lstrip().split(BASE_STOP)[0]
                break
        total = time.perf_counter() - t0
        metrics_monitor_stop()
        text = text.strip()
        hit_limit += r.finish_reason == 'length'
        peak_gb = max(peak_gb, r.peak_memory)
        (out_dir / f'{name}_response_{p.id}.txt').write_text(text + '\n')
        rows.append({
            'prompt_id': p.id, 'model': name, 'family': m.family, 'size': m['size'],
            'input_tokens': len(ids), 'output_tokens': r.generation_tokens, 'output_words': len(text.split()),
            'first_token': json.dumps(first)[1:-1],   # escaped, so newlines stay visible in the CSV
            'time_to_first_token (ms)': round(1000 * ttft, 1),
            'prompt_processing_rate (tokens/s)': round(r.prompt_tps, 1),
            'inference_rate (tokens/s)': round(r.generation_tps, 2),
            'inference_time (s)': round(total, 3),
            'prompt': p.prompt, 'response': text,
        })
        print(f'  [{p.id}] {len(ids)} in -> {r.generation_tokens} out, {r.generation_tps:.1f} tok/s, {total:.1f} s')
    del model, tok
    gc.collect()
    mx.clear_cache()

    df = pd.DataFrame(rows)
    df.to_csv(Path(model_dir) / f'{name}_prompt_metrics.csv', index=False)
    hw = pd.concat([pd.read_csv(Path(sys_dir) / f'{i}_system_metrics.csv') for i in df.prompt_id])
    summary = {
        'model': name, 'family': m.family, 'size': m['size'], 'parameters': m.params, 'precision': 'float16',
        'path': m.path, 'source': m.source, 'prompt_set': PROMPT_SET, 'prompts': len(df),
        'model_load (s)': round(load_s, 1),
        'input_tokens (total)': int(df.input_tokens.sum()),
        'output_tokens (total)': int(df.output_tokens.sum()),
        'output_words (avg)': round(df.output_words.mean()),
        'prompts_hit_token_limit': int(hit_limit),
        'time_to_first_token (ms, avg)': round(df['time_to_first_token (ms)'].mean(), 1),
        'prompt_processing_rate (tokens/s, avg)': round(df['prompt_processing_rate (tokens/s)'].mean(), 1),
        'inference_rate (tokens/s, avg)': round(df['inference_rate (tokens/s)'].mean(), 2),
        'inference_time (s, total)': round(df['inference_time (s)'].sum(), 2),
        'peak_memory (GB)': round(peak_gb, 2),
        'gpu_utilization (%, avg)': round(hw['GPU Utilization (%)'].mean(), 1),
        'gpu_power (W, avg)': round(hw['gpuPower'].mean(), 1),
        'gpu_temperature (C, max)': round(hw['GPU_Temperature(C)'].max(), 1),
        'cpu_utilization (%, avg)': round(hw['CPU_Util(%)'].mean(), 1),
        'cpu_power (W, avg)': round(hw['cpuPower'].mean(), 1),
        'npu_power (W, avg)': round(hw['npuPower'].mean(), 2),
        'system_energy (J)': round(hw['total_system_power'].sum() * 0.1, 1),   # 100 ms samples
    }
    (out_dir / f'{name}_simple_metrics.json').write_text(json.dumps(summary, indent=2))
    return summary, df

summary, per_prompt = run_model(MODEL)
display(per_prompt.drop(columns=['prompt', 'response']))
pd.Series(summary, name=MODEL)

## 5. Compare the family

Runs every other model of `FAMILY` that fits in GPU memory and shows the summaries side by side. Each model runs this
notebook in its own fresh process, so every model starts from the same clean GPU state (in one shared process,
Phi-3-small produces only "!" when another model ran before it). Set `TTT_COMPARE=0` (or `COMPARE_FAMILY = False`
in section 1) to skip.

In [ ]:
import subprocess

results = {MODEL: summary}
if COMPARE_FAMILY:
    for name, m in MODELS[MODELS.family == FAMILY].iterrows():
        if name == MODEL:
            continue
        if not m.fits:
            print(f'{name}: skipped, needs {m.fp16_gb} GB in float16 (GPU budget {GPU_BUDGET_GB:.1f} GB)')
            continue
        print(f'{name}: running in a fresh process...')
        env = {**os.environ, 'TTT_FAMILY': m.family, 'TTT_SIZE': m['size'], 'TTT_PARAMS': m.params,
               'TTT_PROMPTS': PROMPT_SET, 'TTT_MAX_TOKENS': str(MAX_NEW_TOKENS), 'TTT_COMPARE': '0'}
        with tempfile.TemporaryDirectory() as tmp:
            run = subprocess.run([sys.executable, '-m', 'nbconvert', '--to', 'notebook', '--execute',
                                  'notebooks/ttt_text_generation.ipynb', '--ExecutePreprocessor.timeout=-1',
                                  '--output-dir', tmp], env=env, capture_output=True, text=True)
        if run.returncode != 0:
            print(f'  failed:\n{run.stderr[-2000:]}')
            continue
        results[name] = json.loads((Path('outputs/ttt') / m.family / m['size'] / m.params / PROMPT_SET /
                                    f'{name}_simple_metrics.json').read_text())
        print(f"  {results[name]['inference_rate (tokens/s, avg)']} tok/s, {results[name]['inference_time (s, total)']} s")
pd.DataFrame({name: results[name] for name in MODELS.index if name in results})

## 6. All models run so far

Combines every float16 model run on `PROMPT_SET` (any family) into `outputs/ttt/all_models_<prompt set>.csv` and their
per-prompt metrics into `metrics_analysis/ttt/all_models_prompt_metrics_<prompt set>.csv`.

In [ ]:
summaries = [json.loads(f.read_text()) for f in Path('outputs/ttt').glob(f'*/*/*/{PROMPT_SET}/*_simple_metrics.json')]
summaries = sorted((s for s in summaries if s.get('precision') == 'float16'), key=lambda s: list(MODELS.index).index(s['model']))
all_models = pd.DataFrame(summaries)
all_models.to_csv(Path('outputs/ttt') / f'all_models_{PROMPT_SET}.csv', index=False)
all_prompts = pd.concat([
    pd.read_csv(Path('metrics_analysis/ttt') / s['family'] / s['size'] / s['parameters'] / PROMPT_SET /
                'Model_Level_Metrics' / f"{s['model']}_prompt_metrics.csv", dtype={'prompt_id': str})
    for s in summaries])
all_prompts.to_csv(Path('metrics_analysis/ttt') / f'all_models_prompt_metrics_{PROMPT_SET}.csv', index=False)
print(f'{len(all_models)} models, {len(all_prompts)} prompt rows')
all_models